In [71]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import os
pd.options.mode.chained_assignment = None
import optuna
import time
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, train_test_split
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler
import pickle
from collections import defaultdict

In [103]:
path = "saved_files/dataset"
depths = ["in_3_4"]#, "in_1_2", "in_2_3", "in_3_4"]

dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith("_features.csv") and any(depth in archivo for depth in depths):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        # quitamos "_features" al final del nombre
        dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)



# Limpiamos valores nulos
for nombre_df, df in dfs.items():
    for band_set in ["rhow", "rhown","rtoa"]:
        dfs[nombre_df] = df.dropna()

dfs_to_keep = [
    'C2X-Complex_rhow_9x9_depth_in_0_1', 
    'TOA_15x15_depth_in_0_1',
    'C2X-Complex_rhown_9x9_depth_in_0_1',
    'C2X-Complex_rhow_5x5_depth_in_0_1',
    'C2RCC_rhow_5x5_depth_in_0_1',
    'C2RCC_rhow_15x15_depth_in_0_1', 
    'C2X-Complex_rhown_15x15_depth_in_0_1',
    'C2RCC_rhown_5x5_depth_in_0_1', 
    'C2X-Complex_rhow_15x15_depth_in_0_1',
    'C2RCC_rhow_9x9_depth_in_0_1',
    'C2X-Complex_rhow_5x5_depth_in_1_2', 
    'C2X_rhow_3x3_depth_in_1_2',
    'C2X-Complex_rhown_5x5_depth_in_1_2',
    'C2X-Complex_rhow_9x9_depth_in_1_2',
    'C2X-Complex_rhow_3x3_depth_in_1_2',
    'C2X-Complex_rhown_3x3_depth_in_1_2',
    'C2RCC_rhown_3x3_depth_in_1_2',
    'C2X-Complex_rhown_9x9_depth_in_1_2',
    'C2X-Complex_rhow_15x15_depth_in_1_2', 
    'C2X_rhow_5x5_depth_in_1_2',
    'TOA_15x15_depth_in_2_3',
    'TOA_9x9_depth_in_2_3',
    'TOA_5x5_depth_in_2_3', 
    'C2X-Complex_rhow_5x5_depth_in_2_3',
    'C2RCC_rhown_5x5_depth_in_2_3', 
    'TOA_3x3_depth_in_2_3',
    'C2X-Complex_rhown_5x5_depth_in_2_3', 
    'C2RCC_rhow_3x3_depth_in_2_3',
    'C2X-Complex_rhown_9x9_depth_in_2_3', 
    'C2X_rhow_9x9_depth_in_2_3',
    'TOA_9x9_depth_in_3_4',
    'TOA_3x3_depth_in_3_4',
    'TOA_5x5_depth_in_3_4',
    'C2X-Complex_rhow_5x5_depth_in_3_4', 
    'TOA_1x1_depth_in_3_4',
    'TOA_15x15_depth_in_3_4',
    'C2X-Complex_rhown_5x5_depth_in_3_4',
    'C2X-Complex_rhow_9x9_depth_in_3_4',
    'C2X-Complex_rhow_15x15_depth_in_3_4',
    'C2X-Complex_rhown_9x9_depth_in_3_4'
    ]

dfs = {k: dfs[k] for k in dfs_to_keep if k in dfs}


for nombre_df, df in dfs.items():
    # Marcamos las columnas de CHl alta (equivalente a quantile(0.93))
    df["High_Chl"] = df["Chl"]>5
    # Sacamos la estación de cada fecha
    df['Date'] = pd.to_datetime(df['Date'])
    def get_season(month):
        if month in [12, 1, 2]:
            return 'Invierno'
        elif month in [3, 4, 5]:
            return 'Primavera'
        elif month in [6, 7, 8]:
            return 'Verano'
        else:
            return 'Otoño'
    df['Season'] = df['Date'].dt.month.apply(get_season)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')
    df = pd.concat([df.drop(columns=["Season"]),pd.get_dummies(df["Season"])], axis=1)
    dfs[nombre_df] = df

In [60]:
# Plantillas base con los parámetros que no se han optimizado con Optuna
base_params = {
    "XGB": {
        'device': 'cpu',
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'enable_categorical': True,
        'eval_metric': 'rmse'
    },
    "LBM": {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'device': 'cpu',
        'verbosity': -1
    },
    "MLP": {
        'max_iter': 200,
        'shuffle': True,
        'tol': 1e-4,
        'n_iter_no_change': 25,
        'verbose': False,
        'early_stopping': True,
        'validation_fraction': 0.2
    },
    "SVR": {
        'kernel': 'rbf',
        'gamma': 'scale',
        'shrinking': True,
        'tol': 1e-3,
        'max_iter': -1,
        'verbose': False
    },
    "KNN": {
        'weights': 'distance',
        'algorithm': 'auto',
        'metric': 'minkowski',
        'p': 2,
        'n_jobs': -1
    },
    "RF": {
        'criterion': 'squared_error',
        'random_state': 42,
        'verbose': 0
    },
    "CAT": {
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': 42,
        'early_stopping_rounds': 50,
        'verbose': False
    },
    "ELN": {
        'fit_intercept': True,
        'max_iter': 1000,
        'tol': 1e-4,
        'selection': 'cyclic',
        'random_state': 42
    }
}

In [104]:
depth = "in_3_4"

In [106]:
with open(f"training_results/selection_results_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

In [107]:
# Construir el diccionario final
model_params = {}
for (dataset, model), data in results.items():
    best_params = data["best_params"]
    # unimos base + best_params (best_params sobrescribe a base)
    merged = {**base_params.get(model, {}), **best_params}
    if dataset not in model_params:
        model_params[dataset] = {}
    model_params[dataset][model] = merged

In [108]:
def cross_validation_training(dfs, depth):

    FOLDS = 5
    # Para hacer clip a las predicciones y forzar > 0
    correct = True
    results = {}
    rs = 42

    for nombre_df, df in list(dfs.items()):
    #for nombre_df, df in islice(dfs.items(), 3):
        print(f"\n=== Procesando {nombre_df} ===")
        # Ignoramos las columnas de Date, Lat, Lon y Buoy
        df = df.iloc[:, 4:]

        # Separamos el conjunto de datos en train y test: Train 75% Test 25%
        train, test = train_test_split(df, test_size=0.25, random_state=rs, stratify=df["High_Chl"])
        # Seleccionamos la columna que queremos predecir
        target = "Chl"

        # Quitamos esa columna y el indicador de clorofila alta
        X = train.drop(columns=[target, "High_Chl", "Turbidez"])
        # Para y cogemos solamente Chl
        y = train[target]
        # Para poder hacer StratifiedKFold y tener el mismo número de valores de Chl alta en cada fold
        y_class = train["High_Chl"]

        # Definimos X e y para test
        X_test = test.drop(columns=[target, "High_Chl", "Turbidez"])
        y_test = test[target]

        #----Definir modelos usando los params específicos de este dataset----# Esto es diferente a como están en Entrenamiento_V3
        models = {
            "XGB": XGBRegressor(**model_params[nombre_df]["XGB"]),
            "LBM": LGBMRegressor(**model_params[nombre_df]["LBM"]),
            "MLP": MLPRegressor(**model_params[nombre_df]["MLP"]),
            "SVR": SVR(**model_params[nombre_df]["SVR"]),
            "KNN": KNeighborsRegressor(**model_params[nombre_df]["KNN"]),
            "LR": LinearRegression(),
            "RF": RandomForestRegressor(**model_params[nombre_df]["RF"]),
            "CAT": CatBoostRegressor(**model_params[nombre_df]["CAT"]),
            "ELN": ElasticNet(**model_params[nombre_df]["ELN"])
        }


        # Dicts para guardar las predicciones sobre los conjuntos de validación, las y's correspondientes y los índices que corresponden dentro del loop de folds para el ensemble
        val_preds = {name: np.zeros(len(train)) for name in models}
        y_vals = defaultdict(list)
        val_indices = {}
        # Dict para guardar las predicciones sobre test
        test_preds = {name: np.zeros(len(test)) for name in models}
        
        # Dict para guardar resultados
        results[nombre_df] = {name: {'RMSE': [], 'R2': []} for name in models}
        # Stratified KFold de 5 folds
        skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

        # Loop para entrenar cada uno de los modelos
        for name, model in models.items():
            print(f"\n=== Training {name} ===")
            # Loop de folds, manteniendo la proporción de clases (Chl > 5) con y_class
            for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
                print(f"Fold {fold+1}")
                X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

                # Para modelos basados en distancias escalamos los datos
                if name in ["MLP", "SVR", "KNN", "LR", "ELN"]:
                    # Escalado dentro del loop de folds para evitar data leakage entre folds
                    scaler_X = RobustScaler()
                    scaler_y = RobustScaler()
                    X_train_scaled = scaler_X.fit_transform(X_train)
                    X_val_scaled = scaler_X.transform(X_val)
                    X_test_scaled = scaler_X.transform(X_test)
                    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
                    # Entrenamos modelo con datos escalados
                    model.fit(X_train_scaled, y_train_scaled)
                    # Predicción sobre val y test, haciendo la transformada inversa para devolver y a su escala
                    val_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
                    test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()
                # Para modelos basados en árboles no es necesario escalar
                else:
                    # Entrenamos el modelo
                    model.fit(X_train, y_train)
                    # Predicción sobre val y test
                    val_pred = model.predict(X_val)
                    test_pred = model.predict(X_test)

                if correct:
                    val_pred = np.clip(val_pred, 0.2, None)
                    test_pred = np.clip(test_pred, 0.2, None)

                # Guardamos las predicciones sobre val, las y's que les corresponden y los índices
                val_preds[name][val_idx] = val_pred
                if name == list(models.keys())[0]:
                    # Solo lo guardamos una vez
                    y_vals[fold] = y_val
                    val_indices[fold] = val_idx  # val_idx es un array de índices relativos a train

                # Guardamos la predicción de test, haciendo la media entre los folds
                test_preds[name] += test_pred / FOLDS

                # Calculamos y guardamos métricas
                rmse = np.sqrt(mean_squared_error(y_val, val_pred))
                r2 = r2_score(y_val, val_pred)
                results[nombre_df][name]['RMSE'].append(rmse)
                results[nombre_df][name]['R2'].append(r2)

        # Extendemos el dict de resultados con el ensemble
        results[nombre_df]["ENS"] = {'RMSE': [], 'R2': []}

        for fold in range(FOLDS):
            # Índices y valores del fold actual
            fold_val_idx = val_indices[fold]
            meta_X_val = np.vstack([val_preds[model][fold_val_idx] for model in models]).T
            meta_y_val = y_vals[fold]

            # Índices de entrenamiento: todos menos el fold actual
            train_folds = [i for i in range(FOLDS) if i != fold]
            train_idx = np.concatenate([val_indices[i] for i in train_folds])
            meta_X_train = np.vstack([val_preds[model][train_idx] for model in models]).T
            meta_y_train = y.iloc[train_idx]

            # Entrenamos el meta-modelo solo con los otros 4 folds
            meta_model = Ridge().fit(meta_X_train, meta_y_train)

            # Predicción en el fold actual (no visto)
            ensemble_pred = meta_model.predict(meta_X_val)
            if correct:
                ensemble_pred = np.clip(ensemble_pred, 0.2, None)

            rmse = np.sqrt(mean_squared_error(meta_y_val, ensemble_pred))
            r2 = r2_score(meta_y_val, ensemble_pred)
            results[nombre_df]["ENS"]['RMSE'].append(rmse)
            results[nombre_df]["ENS"]['R2'].append(r2)



    # === Evaluación final sobre test ===
        for name in models:
            rmse_test = np.sqrt(mean_squared_error(y_test, test_preds[name]))
            r2_test = r2_score(y_test, test_preds[name])
            results[nombre_df][name]["RMSE test"] = rmse_test
            results[nombre_df][name]["R2 test"] = r2_test

        # Construcción del meta-modelo sobre todo el conjunto de validación
        final_meta_X = np.vstack([val_preds[model] for model in models]).T
        final_meta_y = y.values
        ensemble_model = Ridge().fit(final_meta_X, final_meta_y)

        # Predicción sobre test del ensemble
        meta_X_test = np.vstack([test_preds[model] for model in models]).T
        ensemble_test_pred = ensemble_model.predict(meta_X_test)
        if correct:
            ensemble_test_pred = np.clip(ensemble_test_pred, 0.3, None)
        # Evaluación del ensemble sobre test
        rmse_ens_test = np.sqrt(mean_squared_error(y_test, ensemble_test_pred))
        r2_ens_test = r2_score(y_test, ensemble_test_pred)
        results[nombre_df]["ENS"]["RMSE test"] = rmse_ens_test
        results[nombre_df]["ENS"]["R2 test"] = r2_ens_test

    with open(f"training_results/results_entrenamiento_final_{depth}.pkl", "wb") as f:
        pickle.dump(results, f)

    return results

In [109]:
depths[0]

'in_3_4'

In [110]:
results = cross_validation_training(dfs, depths[0])


=== Procesando TOA_9x9_depth_in_3_4 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando TOA_3x3_depth_in_3_4 ===

=== Training XGB ===
Fold 1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.560e+01, tolerance: 4.075e-02
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.133e+02, tolerance: 4.583e-02
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations

Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando TOA_5x5_depth_in_3_4 ===

=== Training XGB ===
Fold 1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.417e+01, tolerance: 4.075e-02
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.101e+02, tolerance: 4.583e-02
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations

Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2X-Complex_rhow_5x5_depth_in_3_4 ===

=== Training XGB ===
Fold 1


...

Fold 4


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 5


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


In [111]:
with open(f"training_results/results_entrenamiento_final_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

In [112]:
rows = []

for df_name, model_scores in results.items():
    row = {}
    for model_name, metrics in model_scores.items():
        for metric_name, values in metrics.items():
            if isinstance(values, list):  # Solo para los que tienen listas (folds)
                mean_val = np.mean(values)
                std_val = np.std(values)
                row[(metric_name, model_name)] = f"{mean_val:.2f} ± {std_val:.2f}"
            else:
                # Para el ensemble que tiene un único valor
                row[(metric_name, model_name)] = f"{values:.2f}"
    rows.append((df_name, row))

df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
df_results = df_results.sort_index(axis=1, level=0)
df_results = df_results.sort_index(axis=0)

In [114]:
df_sorted = df_results["R2 test"].copy()
# Añadir una columna auxiliar con el R2 máximo por fila
df_sorted["max_R2"] = df_sorted.max(axis=1)
# Ordenar por esa columna en orden descendente
df_sorted = df_sorted.sort_values("max_R2", ascending=False)
# Eliminar la columna auxiliar
df_sorted = df_sorted.drop(columns="max_R2")
df_sorted

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_15x15_depth_in_3_4,0.66,0.50,0.58,0.62,0.59,0.38,0.61,0.61,0.47,0.57
TOA_3x3_depth_in_3_4,0.59,0.46,0.55,0.61,0.64,0.40,0.60,0.66,0.45,0.61
C2X-Complex_rhow_5x5_depth_in_3_4,0.59,0.50,0.52,0.57,0.62,0.41,0.59,0.65,0.51,0.63
TOA_5x5_depth_in_3_4,0.58,0.44,0.50,0.59,0.64,0.45,0.58,0.63,0.42,0.62
C2X-Complex_rhown_5x5_depth_in_3_4,0.60,0.56,0.57,0.57,0.59,0.34,0.55,0.63,0.51,0.61
TOA_9x9_depth_in_3_4,0.61,0.46,0.63,0.62,0.59,0.38,0.62,0.61,0.47,0.58
C2X-Complex_rhow_15x15_depth_in_3_4,0.49,0.60,0.43,0.42,0.46,0.41,0.62,0.55,0.54,0.50
C2X-Complex_rhow_9x9_depth_in_3_4,0.55,0.56,0.51,0.55,0.57,0.46,0.61,0.62,0.58,0.60
TOA_1x1_depth_in_3_4,0.54,0.38,0.44,0.60,0.49,0.18,0.54,0.55,0.44,0.46
C2X-Complex_rhown_9x9_depth_in_3_4,0.59,0.58,0.50,0.52,0.54,0.37,0.57,0.59,0.55,0.58


### Evaluación de resultados

In [120]:
def create_df_results(results):
    rows = []
    for df_name, model_scores in results.items():
        row = {}
        for model_name, metrics in model_scores.items():
            for metric_name, values in metrics.items():
                if isinstance(values, list):  # Solo para los que tienen listas (folds)
                    mean_val = np.mean(values)
                    std_val = np.std(values)
                    row[(metric_name, model_name)] = f"{mean_val:.2f} ± {std_val:.2f}"
                else:
                    # Para el ensemble que tiene un único valor
                    row[(metric_name, model_name)] = f"{values:.2f}"
        rows.append((df_name, row))

    df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
    df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
    df_results = df_results.sort_index(axis=1, level=0)
    df_results = df_results.sort_index(axis=0)

    df_sorted = df_results["R2 test"].copy()
    df_sorted["max_R2"] = df_sorted.max(axis=1)
    df_sorted = df_sorted.sort_values("max_R2", ascending=False)
    df_sorted_test = df_sorted.drop(columns="max_R2")

    df_sorted = df_results["R2"].copy()
    df_sorted["max_R2"] = df_sorted.max(axis=1)
    df_sorted = df_sorted.sort_values("max_R2", ascending=False)
    df_sorted_train = df_sorted.drop(columns="max_R2")

    return df_sorted_train, df_sorted_test

#### Profundidad 0-1

In [122]:
depth = "in_0_1"

with open(f"training_results/results_entrenamiento_final_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

results_train, results_test = create_df_results(results)

In [124]:
results_train

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2RCC_rhow_9x9_depth_in_0_1,0.76 ± 0.08,0.58 ± 0.09,0.71 ± 0.10,0.76 ± 0.14,0.63 ± 0.09,0.55 ± 0.10,0.59 ± 0.24,0.67 ± 0.08,0.72 ± 0.06,0.71 ± 0.08
C2RCC_rhow_5x5_depth_in_0_1,0.74 ± 0.08,0.59 ± 0.11,0.61 ± 0.15,0.74 ± 0.08,0.69 ± 0.14,0.55 ± 0.16,0.76 ± 0.05,0.67 ± 0.09,0.73 ± 0.04,0.71 ± 0.08
C2RCC_rhown_5x5_depth_in_0_1,0.75 ± 0.09,0.59 ± 0.12,0.64 ± 0.08,0.73 ± 0.08,0.73 ± 0.09,0.54 ± 0.21,0.66 ± 0.12,0.67 ± 0.05,0.73 ± 0.03,0.74 ± 0.08
C2X-Complex_rhow_15x15_depth_in_0_1,0.74 ± 0.13,0.58 ± 0.11,0.73 ± 0.09,0.70 ± 0.11,0.69 ± 0.12,0.05 ± 0.95,0.69 ± 0.17,0.63 ± 0.13,0.70 ± 0.14,0.69 ± 0.12
C2X-Complex_rhown_15x15_depth_in_0_1,0.74 ± 0.11,0.59 ± 0.12,0.63 ± 0.11,0.71 ± 0.10,0.67 ± 0.11,0.54 ± 0.16,0.61 ± 0.18,0.64 ± 0.13,0.68 ± 0.13,0.69 ± 0.12
TOA_15x15_depth_in_0_1,0.68 ± 0.15,0.49 ± 0.10,0.72 ± 0.10,0.73 ± 0.11,0.59 ± 0.11,0.30 ± 0.26,0.51 ± 0.14,0.67 ± 0.13,0.34 ± 0.03,0.65 ± 0.09
C2RCC_rhow_15x15_depth_in_0_1,0.70 ± 0.16,0.51 ± 0.11,0.61 ± 0.19,0.72 ± 0.08,0.57 ± 0.28,0.25 ± 0.81,0.67 ± 0.22,0.61 ± 0.22,0.68 ± 0.11,0.63 ± 0.20
C2X-Complex_rhow_9x9_depth_in_0_1,0.71 ± 0.09,0.48 ± 0.15,0.65 ± 0.10,0.61 ± 0.12,0.63 ± 0.10,0.40 ± 0.32,0.63 ± 0.13,0.60 ± 0.15,0.67 ± 0.10,0.66 ± 0.09
C2X-Complex_rhown_9x9_depth_in_0_1,0.69 ± 0.10,0.53 ± 0.11,0.65 ± 0.13,0.63 ± 0.13,0.52 ± 0.25,0.42 ± 0.27,0.67 ± 0.14,0.54 ± 0.19,0.68 ± 0.08,0.59 ± 0.18
C2X-Complex_rhow_5x5_depth_in_0_1,0.66 ± 0.11,0.49 ± 0.17,0.65 ± 0.14,0.66 ± 0.12,0.52 ± 0.17,0.38 ± 0.19,0.63 ± 0.13,0.55 ± 0.17,0.68 ± 0.08,0.52 ± 0.21


In [125]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_9x9_depth_in_0_1,0.88,0.68,0.89,0.76,0.87,0.65,0.78,0.82,0.74,0.89
C2X-Complex_rhown_9x9_depth_in_0_1,0.86,0.70,0.86,0.78,0.85,0.65,0.82,0.80,0.74,0.84
C2RCC_rhow_5x5_depth_in_0_1,0.80,0.64,0.82,0.85,0.76,0.43,0.79,0.75,0.71,0.81
C2X-Complex_rhow_15x15_depth_in_0_1,0.82,0.68,0.84,0.85,0.81,0.68,0.78,0.79,0.72,0.81
TOA_15x15_depth_in_0_1,0.68,0.56,0.85,0.85,0.61,0.51,0.69,0.49,0.34,0.61
C2RCC_rhow_15x15_depth_in_0_1,0.77,0.58,0.84,0.84,0.71,0.61,0.77,0.76,0.70,0.78
C2RCC_rhown_5x5_depth_in_0_1,0.82,0.64,0.84,0.82,0.80,0.50,0.81,0.78,0.71,0.84
C2X-Complex_rhow_5x5_depth_in_0_1,0.84,0.53,0.76,0.77,0.84,-0.11,0.74,0.80,0.72,0.84
C2RCC_rhow_9x9_depth_in_0_1,0.81,0.67,0.80,0.79,0.80,0.65,0.74,0.76,0.72,0.83
C2X-Complex_rhown_15x15_depth_in_0_1,0.77,0.67,0.79,0.81,0.73,0.64,0.73,0.73,0.71,0.77


#### Profundidad 1-2

In [126]:
depth = "in_1_2"

with open(f"training_results/results_entrenamiento_final_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

results_train, results_test = create_df_results(results)

In [127]:
results_train

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_15x15_depth_in_1_2,0.68 ± 0.14,0.56 ± 0.24,0.69 ± 0.14,0.71 ± 0.14,0.59 ± 0.12,0.51 ± 0.23,0.69 ± 0.11,0.57 ± 0.19,0.75 ± 0.05,0.60 ± 0.18
C2X-Complex_rhown_9x9_depth_in_1_2,0.67 ± 0.05,0.59 ± 0.08,0.67 ± 0.08,0.65 ± 0.05,0.67 ± 0.11,0.36 ± 0.43,0.59 ± 0.16,0.64 ± 0.08,0.72 ± 0.06,0.68 ± 0.10
C2X-Complex_rhow_9x9_depth_in_1_2,0.69 ± 0.02,0.58 ± 0.08,0.67 ± 0.09,0.61 ± 0.06,0.65 ± 0.09,0.36 ± 0.32,0.59 ± 0.11,0.61 ± 0.09,0.71 ± 0.07,0.67 ± 0.09
C2RCC_rhown_3x3_depth_in_1_2,0.66 ± 0.06,0.55 ± 0.10,0.56 ± 0.14,0.67 ± 0.04,0.47 ± 0.27,0.35 ± 0.38,0.70 ± 0.06,0.55 ± 0.09,0.67 ± 0.15,0.57 ± 0.08
C2X-Complex_rhow_5x5_depth_in_1_2,0.63 ± 0.07,0.47 ± 0.22,0.59 ± 0.07,0.63 ± 0.12,0.60 ± 0.10,-0.06 ± 0.84,0.59 ± 0.04,0.51 ± 0.14,0.69 ± 0.07,0.59 ± 0.14
C2X-Complex_rhown_3x3_depth_in_1_2,0.67 ± 0.06,0.42 ± 0.29,0.57 ± 0.14,0.59 ± 0.13,0.55 ± 0.05,-0.57 ± 2.27,0.60 ± 0.15,0.56 ± 0.08,0.58 ± 0.21,0.64 ± 0.06
C2X-Complex_rhown_5x5_depth_in_1_2,0.61 ± 0.09,0.48 ± 0.21,0.54 ± 0.10,0.60 ± 0.12,0.51 ± 0.18,-0.61 ± 1.73,0.61 ± 0.08,0.51 ± 0.13,0.67 ± 0.06,0.57 ± 0.12
C2X-Complex_rhow_3x3_depth_in_1_2,0.66 ± 0.09,0.45 ± 0.23,0.56 ± 0.07,0.59 ± 0.12,0.60 ± 0.08,-0.23 ± 1.59,0.61 ± 0.08,0.61 ± 0.10,0.61 ± 0.15,0.63 ± 0.13
C2X_rhow_5x5_depth_in_1_2,0.52 ± 0.08,0.49 ± 0.18,0.43 ± 0.09,0.57 ± 0.10,0.39 ± 0.23,0.05 ± 0.84,0.54 ± 0.09,0.47 ± 0.08,0.59 ± 0.07,0.41 ± 0.10
C2X_rhow_3x3_depth_in_1_2,0.50 ± 0.10,0.51 ± 0.18,0.48 ± 0.16,0.52 ± 0.13,0.39 ± 0.09,0.30 ± 0.27,0.53 ± 0.10,0.49 ± 0.10,0.56 ± 0.10,0.46 ± 0.10


In [128]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_5x5_depth_in_1_2,0.86,0.60,0.87,0.77,0.82,0.71,0.76,0.73,0.86,0.83
C2X-Complex_rhow_9x9_depth_in_1_2,0.86,0.55,0.84,0.76,0.80,0.53,0.66,0.79,0.83,0.83
C2X-Complex_rhown_5x5_depth_in_1_2,0.86,0.63,0.86,0.78,0.80,0.71,0.74,0.71,0.86,0.78
C2X_rhow_3x3_depth_in_1_2,0.82,0.71,0.62,0.75,0.84,0.67,0.75,0.77,0.63,0.83
C2X-Complex_rhow_3x3_depth_in_1_2,0.80,0.56,0.80,0.79,0.76,0.54,0.72,0.69,0.83,0.77
C2X-Complex_rhown_3x3_depth_in_1_2,0.82,0.59,0.83,0.75,0.74,0.67,0.74,0.70,0.78,0.77
C2X-Complex_rhown_9x9_depth_in_1_2,0.81,0.59,0.79,0.77,0.71,0.48,0.76,0.70,0.83,0.73
C2RCC_rhown_3x3_depth_in_1_2,0.81,0.60,0.81,0.81,0.76,0.62,0.79,0.58,0.78,0.76
C2X-Complex_rhow_15x15_depth_in_1_2,0.81,0.65,0.70,0.74,0.81,-0.44,0.75,0.80,0.77,0.78
C2X_rhow_5x5_depth_in_1_2,0.80,0.72,0.64,0.74,0.79,0.72,0.65,0.68,0.66,0.81


#### Profundidad 2-3

In [129]:
depth = "in_2_3"

with open(f"training_results/results_entrenamiento_final_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

results_train, results_test = create_df_results(results)

In [130]:
results_train

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_9x9_depth_in_2_3,0.64 ± 0.09,0.44 ± 0.09,0.63 ± 0.11,0.69 ± 0.08,0.58 ± 0.14,0.37 ± 0.16,0.53 ± 0.19,0.57 ± 0.09,0.48 ± 0.04,0.59 ± 0.11
TOA_15x15_depth_in_2_3,0.65 ± 0.09,0.46 ± 0.07,0.66 ± 0.11,0.66 ± 0.10,0.50 ± 0.29,0.41 ± 0.13,0.50 ± 0.21,0.53 ± 0.09,0.48 ± 0.05,0.50 ± 0.20
C2X-Complex_rhown_5x5_depth_in_2_3,0.54 ± 0.26,0.49 ± 0.23,0.52 ± 0.29,0.60 ± 0.13,0.46 ± 0.26,0.22 ± 0.65,0.52 ± 0.22,0.50 ± 0.27,0.65 ± 0.07,0.51 ± 0.25
C2X-Complex_rhown_9x9_depth_in_2_3,0.49 ± 0.24,0.46 ± 0.24,0.49 ± 0.24,0.57 ± 0.15,0.47 ± 0.28,0.26 ± 0.65,0.49 ± 0.20,0.51 ± 0.23,0.64 ± 0.10,0.51 ± 0.23
TOA_5x5_depth_in_2_3,0.60 ± 0.10,0.47 ± 0.08,0.63 ± 0.12,0.60 ± 0.14,0.59 ± 0.12,0.43 ± 0.13,0.61 ± 0.07,0.46 ± 0.18,0.48 ± 0.06,0.57 ± 0.13
C2RCC_rhown_5x5_depth_in_2_3,0.60 ± 0.19,0.58 ± 0.13,0.53 ± 0.19,0.60 ± 0.20,0.59 ± 0.18,0.52 ± 0.18,0.63 ± 0.10,0.58 ± 0.19,0.61 ± 0.09,0.60 ± 0.15
C2X-Complex_rhow_5x5_depth_in_2_3,0.54 ± 0.24,0.48 ± 0.24,0.47 ± 0.40,0.60 ± 0.13,0.46 ± 0.31,0.20 ± 0.70,0.57 ± 0.12,0.51 ± 0.23,0.62 ± 0.12,0.51 ± 0.28
C2RCC_rhow_3x3_depth_in_2_3,0.55 ± 0.22,0.50 ± 0.17,0.55 ± 0.15,0.55 ± 0.21,0.45 ± 0.26,0.38 ± 0.22,0.57 ± 0.14,0.51 ± 0.26,0.62 ± 0.07,0.46 ± 0.30
TOA_3x3_depth_in_2_3,0.57 ± 0.12,0.46 ± 0.09,0.60 ± 0.14,0.58 ± 0.16,0.55 ± 0.11,0.37 ± 0.13,0.61 ± 0.10,0.50 ± 0.15,0.46 ± 0.06,0.53 ± 0.19
C2X_rhow_9x9_depth_in_2_3,0.54 ± 0.22,0.52 ± 0.25,0.48 ± 0.30,0.56 ± 0.14,0.44 ± 0.22,0.35 ± 0.39,0.60 ± 0.10,0.54 ± 0.16,0.51 ± 0.25,0.47 ± 0.28


In [131]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_15x15_depth_in_2_3,0.73,0.40,0.81,0.78,0.65,0.21,0.63,0.62,0.52,0.66
TOA_9x9_depth_in_2_3,0.71,0.39,0.79,0.78,0.67,0.27,0.65,0.63,0.52,0.63
TOA_3x3_depth_in_2_3,0.74,0.43,0.76,0.71,0.72,0.35,0.72,0.65,0.48,0.69
TOA_5x5_depth_in_2_3,0.75,0.41,0.76,0.75,0.72,0.36,0.69,0.62,0.50,0.73
C2RCC_rhown_5x5_depth_in_2_3,0.71,0.58,0.73,0.70,0.67,0.61,0.69,0.64,0.65,0.68
C2X-Complex_rhown_9x9_depth_in_2_3,0.73,0.44,0.61,0.58,0.65,0.41,0.68,0.64,0.68,0.66
C2X-Complex_rhow_5x5_depth_in_2_3,0.72,0.16,0.71,0.59,0.63,0.24,0.64,0.55,0.72,0.65
C2X-Complex_rhown_5x5_depth_in_2_3,0.68,0.18,0.60,0.58,0.58,-0.48,0.68,0.54,0.72,0.61
C2RCC_rhow_3x3_depth_in_2_3,0.68,0.54,0.68,0.67,0.59,0.55,0.69,0.61,0.63,0.59
C2X_rhow_9x9_depth_in_2_3,0.69,0.62,0.62,0.58,0.64,0.64,0.63,0.63,0.67,0.65


#### Profundidad 3-4

In [132]:
depth = "in_3_4"

with open(f"training_results/results_entrenamiento_final_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

results_train, results_test = create_df_results(results)

In [133]:
results_train

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_15x15_depth_in_3_4,0.55 ± 0.03,0.38 ± 0.16,0.46 ± 0.18,0.52 ± 0.14,0.47 ± 0.04,0.30 ± 0.18,0.56 ± 0.05,0.39 ± 0.16,0.53 ± 0.16,0.45 ± 0.06
TOA_9x9_depth_in_3_4,0.54 ± 0.03,0.39 ± 0.14,0.39 ± 0.26,0.51 ± 0.16,0.49 ± 0.05,0.38 ± 0.17,0.48 ± 0.13,0.42 ± 0.17,0.53 ± 0.17,0.51 ± 0.05
TOA_5x5_depth_in_3_4,0.53 ± 0.05,0.40 ± 0.18,0.41 ± 0.23,0.43 ± 0.14,0.45 ± 0.09,0.32 ± 0.25,0.40 ± 0.14,0.36 ± 0.21,0.52 ± 0.16,0.46 ± 0.10
C2X-Complex_rhow_15x15_depth_in_3_4,0.44 ± 0.20,0.35 ± 0.20,0.42 ± 0.34,0.50 ± 0.19,0.31 ± 0.27,0.12 ± 0.36,0.39 ± 0.17,0.40 ± 0.24,0.51 ± 0.14,0.39 ± 0.24
C2X-Complex_rhow_9x9_depth_in_3_4,0.45 ± 0.12,0.38 ± 0.21,0.38 ± 0.28,0.43 ± 0.21,0.33 ± 0.20,0.30 ± 0.17,0.21 ± 0.55,0.43 ± 0.17,0.51 ± 0.13,0.38 ± 0.19
TOA_1x1_depth_in_3_4,0.46 ± 0.08,0.39 ± 0.18,0.35 ± 0.30,0.42 ± 0.13,0.33 ± 0.15,0.39 ± 0.17,0.46 ± 0.05,0.32 ± 0.14,0.50 ± 0.17,0.38 ± 0.13
TOA_3x3_depth_in_3_4,0.43 ± 0.10,0.40 ± 0.15,0.34 ± 0.30,0.43 ± 0.10,0.35 ± 0.14,0.31 ± 0.18,0.47 ± 0.08,0.28 ± 0.19,0.50 ± 0.16,0.40 ± 0.10
C2X-Complex_rhow_5x5_depth_in_3_4,0.48 ± 0.18,0.37 ± 0.20,0.36 ± 0.30,0.40 ± 0.18,0.40 ± 0.15,0.26 ± 0.28,0.44 ± 0.24,0.40 ± 0.24,0.43 ± 0.14,0.41 ± 0.21
C2X-Complex_rhown_9x9_depth_in_3_4,0.38 ± 0.14,0.38 ± 0.12,0.40 ± 0.21,0.42 ± 0.20,0.38 ± 0.14,0.31 ± 0.22,0.43 ± 0.12,0.39 ± 0.20,0.46 ± 0.12,0.33 ± 0.15
C2X-Complex_rhown_5x5_depth_in_3_4,0.43 ± 0.15,0.37 ± 0.17,0.32 ± 0.22,0.42 ± 0.20,0.33 ± 0.37,0.30 ± 0.21,0.41 ± 0.18,0.38 ± 0.27,0.44 ± 0.14,0.35 ± 0.27


In [134]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_15x15_depth_in_3_4,0.66,0.50,0.58,0.62,0.59,0.38,0.61,0.61,0.47,0.57
TOA_3x3_depth_in_3_4,0.59,0.46,0.55,0.61,0.64,0.40,0.60,0.66,0.45,0.61
C2X-Complex_rhow_5x5_depth_in_3_4,0.59,0.50,0.52,0.57,0.62,0.41,0.59,0.65,0.51,0.63
TOA_5x5_depth_in_3_4,0.58,0.44,0.50,0.59,0.64,0.45,0.58,0.63,0.42,0.62
C2X-Complex_rhown_5x5_depth_in_3_4,0.60,0.56,0.57,0.57,0.59,0.34,0.55,0.63,0.51,0.61
TOA_9x9_depth_in_3_4,0.61,0.46,0.63,0.62,0.59,0.38,0.62,0.61,0.47,0.58
C2X-Complex_rhow_15x15_depth_in_3_4,0.49,0.60,0.43,0.42,0.46,0.41,0.62,0.55,0.54,0.50
C2X-Complex_rhow_9x9_depth_in_3_4,0.55,0.56,0.51,0.55,0.57,0.46,0.61,0.62,0.58,0.60
TOA_1x1_depth_in_3_4,0.54,0.38,0.44,0.60,0.49,0.18,0.54,0.55,0.44,0.46
C2X-Complex_rhown_9x9_depth_in_3_4,0.59,0.58,0.50,0.52,0.54,0.37,0.57,0.59,0.55,0.58
